# OMRモデルの学習

In [1]:
import yaml
import torch
from yolov3.models.yolo import BaseModel
from src.domain.model import OMRModel
from src.domain.dataloader import CustomDataset, custom_collate_fn
from src.domain.loss import CustomLoss
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.optim as optim
from torchvision import transforms

/home/docker/.cache/pypoetry/virtualenvs/repo-uZbbGesQ-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_dir = "../models/"
config_path = model_dir + "config/omr_yolov5s.yaml"  #'yolov3/models/yolov5s.yaml'
model_path = model_dir + "pre_trained/yolov5s.pt"

data_dir = "../data/"
hyp_path = data_dir + "hyps/hyp.scratch-low.yaml"
img_dir = data_dir + "dense/images/"
annotation_dir = data_dir + "dense/labels_mapping_to_center_and_normalization/"


output_path = model_dir + "fine_tuned/omr_yolov5s_finetuned_resize.pth"

In [5]:
model = OMRModel(config_path)
# 事前学習済みモデルの読み込み
pretrained_dict = torch.load(model_path)["model"].state_dict()
model_dict = model.state_dict()

# 最終層以外のパラメータのみを適用
pretrained_dict = {
    k: v
    for k, v in pretrained_dict.items()
    if k in model_dict and not k.startswith("model.24")
}  # model.24は最終層
model_dict.update(pretrained_dict)
model.load_state_dict(model_dict)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
model.to(device)

# ハイパラの設定
with open(hyp_path, errors="ignore") as f:
    hyp = yaml.safe_load(f)
model.hyp = hyp

# カスタム損失関数
criterion = CustomLoss(model)

# オプティマイザ
optimizer = optim.Adam(model.parameters(), lr=0.002)

# データローダ
h_resize = 800
w_resize = int(h_resize * 1)
train_dataset = CustomDataset(
    img_dir=img_dir,
    annotation_dir=annotation_dir,
    transform=transforms.Compose(
        [
            transforms.Resize((h_resize, w_resize)),  # h,w
            transforms.ToTensor(),
        ]
    ),
)
train_loader = DataLoader(
    train_dataset, batch_size=8, shuffle=True, collate_fn=custom_collate_fn
)


                 from  n    params  module                                  arguments                     
  0                -1  1      3520  yolov3.models.common.Conv               [3, 32, 6, 2, 2]              
  1                -1  1     18560  yolov3.models.common.Conv               [32, 64, 3, 2]                
  2                -1  1     18816  yolov3.models.common.C3                 [64, 64, 1]                   
  3                -1  1     73984  yolov3.models.common.Conv               [64, 128, 3, 2]               
  4                -1  2    115712  yolov3.models.common.C3                 [128, 128, 2]                 
  5                -1  1    295424  yolov3.models.common.Conv               [128, 256, 3, 2]              
  6                -1  3    625152  yolov3.models.common.C3                 [256, 256, 3]                 
  7                -1  1   1180672  yolov3.models.common.Conv               [256, 512, 3, 2]              
  8                -1  1   1182720  

device: cuda


In [6]:
# tqdmの表示フォーマット
TQDM_BAR_FORMAT = "{l_bar}{bar:10}{r_bar}"

# トレーニングループ
num_epochs = 1

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, total=len(train_loader), bar_format=TQDM_BAR_FORMAT)
    i = 0
    for images, targets in pbar:
        i += 1

        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss, loss_items = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        running_loss += loss_value
        pbar.set_description(
            f"Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / i}"
        )

        # GPUメモリの解放
        del images, targets, outputs, loss, loss_items
        torch.cuda.empty_cache()

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}")


# トレーニング済みモデルの保存
torch.save(model.state_dict(), output_path)
print("model save!")

Epoch [1/1], Loss: 2.2300747222678607: 100%|██████████| 215/215 [06:18<00:00,  1.76s/it]

Epoch [1/1], Loss: 2.2300747222678607
model save!
